# Selection Analysis -- how many clusters survive each cut

Reads the charge-light-matching files/events exactly the way
`HighStats_ChargeLightMatching_Evaluation_MultiFile.ipynb` does (same
`PARENT_DIR`, same file/event discovery, same `target_file` /
`target_event` / `target_file_range` selectors, same selection parameters),
then counts how many CLUSTERS survive each successive selection and draws
the result as horizontal bar blocks -- one block per cut stage, top block =
before any cut.

**True side** (truth available, `q_true>0` = neutrino) -- two bars per block,
Cosmic / Neutrino:

1. No cuts
2. True energy >= `min_cluster_energy` (100 MeV)
3. + beam window
4. + dead area & wire-readout sensitive plane

The flow starts at the energy cut, not at the raw uncut clusters: uncut, an
event carries ~175 true clusters, nearly all tiny sub-threshold depositions
(20356 before the energy cut vs 76 after the beam window over 118 events), which
swamps every later stage. Starting here also makes the first block the same
population `near_miss_investigation.py` counts, so the two analyses agree.

**Reco side** -- one bar per block (Total only; reco clusters carry no truth
label, so there is no cosmic/neutrino split to draw):

1. No cuts
2. + beam window
3. + dead area & wire-readout sensitive plane

All stages above are **counted** and written to `selection_flow_counts.txt`.
The **drawn** bars are a subset: the true plot shows No cuts + the energy cut,
and the reco plot shows No cuts + beam window. The true beam-window block is not
drawn because "in beam window" is not a truth quantity -- true clusters carry no
flash and no time, so the label is inferred by spatially matching to a reco
cluster whose flash fell in the window, mixing beam timing with reconstruction
efficiency. The dead-area / wire-readout block is not drawn on either side (it
removed nothing at either level).

Cuts are cumulative: every stage is applied on top of the clusters that
survived the stage above it. Bars are labelled with the raw cluster count and,
from the second stage down, the survival percentage relative to that same
category's no-cut count.

Drawing lives in `DrawRecoTrueClusterCount.py`; the counting lives in the main
loop below, next to the cuts it is counting.

In [ ]:
# Charge-light matching is a combined-APA evaluation (img-global / sed-sce are
# already global across APAs) -- no per-APA/face looping like the older pipeline.
files     = "all"   # "all", or 1/2/3/... to limit the number of file subdirectories processed
events    = "all"   # "all", or 1/2/3/... to limit the number of events processed per file

# ========================================================================
# SELECTIVE FILE/EVENT FILTERING (Optional)
# ========================================================================
# Set to None to process all files/events, or specify to run only specific ones
# Example: target_file = "file0", target_event = 3  (to process only file0, event 3)
target_file  = None  # Set to "file0", "file1", etc. to process specific file only
target_event = None   # Set to event number (0, 1, ..., 9) to process specific event only

# Range of file INDICES to run, inclusive on both ends: (6, 9) runs file6, file7,
# file8, file9. None runs every file. Matched on the number at the end of the
# directory name, NOT on position in the list -- the directories sort
# lexicographically (file0, file1, file10, file11, file2, ...), so a positional
# slice would pick the wrong files. This is the only file selector that survives
# that sort order; the `files = N` knob above still takes the first N in
# lexicographic order.
# NOTE: target_file (above) is applied too, so set it to None when using a range,
# otherwise only the one file that satisfies both runs.
target_file_range = None   # e.g. (6, 9) for file6..file9


# ========================================================================
# Decide which plots to draw
# ========================================================================
b_draw_event_level_plots = False   # Draw event-level plots (one per event)
b_draw_file_level_plots  = True   # Draw file-level plots (one per file)
b_draw_job_level_plots   = True   # Draw job-level plots (one per job)

In [ ]:
%load_ext autoreload
%autoreload 2

# python libraries
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
import sys
import os
from scipy.spatial import KDTree
import pandas as pd
import seaborn as sns
import time
import io
import contextlib
from datetime import datetime, timedelta

np.set_printoptions(linewidth=1000)

# Record job start time (used to report total job runtime at the end)
job_start_time = time.time()
print(f"Job started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


In [ ]:
# Import functions from Python modules.
# Same reading/selection functions as HighStats_ChargeLightMatching_Evaluation_MultiFile.ipynb
# (identical cut implementations -- this notebook only counts what survives them,
# it does not redefine any cut), plus the new selection-flow drawers.
from readfiles import ensure_data_extracted, read_charge_light_files_for_event, flatten_mc_tree
from selections import (
    GroupClustersByID, build_true_points_charge_light, apply_deadarea_cut_true_charge_light,
    reassign_cluster_ID_true_charge_light,
    apply_energy_cutoff,
    apply_wire_readout_sensitive_yz_plane_cut_true, apply_wire_readout_sensitive_yz_plane_cut_reco,
)
from efficiency_purity_estimate import EvaluateEfficiency
from metadata import build_cluster_flash_metadata, build_img_cluster_flash_metadata, build_neutrino_vertex_records
from DrawRecoTrueFlashes import BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US
from DrawRecoTrueClusterCount import (
    DrawTrueClusterSelectionFlow, DrawRecoClusterSelectionFlow, write_selection_flow_table,
)

In [ ]:
# Configuration: Parent directory containing multiple file subdirectories (file0/, file1/, ...)
#
# Expected structure (per file subdirectory):
# PARENT_DIR/
#   file0/mabc.zip                      <- shipped as a single zip, extracted once into file0/data/
#   file0/data/0/0-img-global.json                      (reco clusters, combined APA)
#   file0/data/0/0-sed-sce_drift_smear_readout.json      (true clusters, combined APA)
#   file0/data/0/0-mc.json                               (particle truth ancestry tree)
#   file0/data/0/0-op.json                               (optical/light info)
#   file0/data/1/, 2/, ... (one subdirectory per event)
#   file1/mabc.zip, file1/data/..., etc.

PARENT_DIR = Path("Haiwang_files_charge_light_matching_MCP2025C_Fall_production")

# Number of files to process (convert 'files' variable to num_files_to_process)
num_files_to_process = None if files == "all" else files

# Number of events to process (convert 'events' variable to num_events_to_process)
num_events_to_process = None if events == "all" else events

# Output directory for plots
PLOTBASEDIR = Path("multi_file_plots_charge_light_matching")
PLOTBASEDIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"Parent directory: {PARENT_DIR}")
print(f"Plot base directory: {PLOTBASEDIR}")
print(f"Files to process: {files}")
print(f"Events to process: {events}")

if target_file is not None or target_event is not None or target_file_range is not None:
    print(f"\n⚡ SELECTIVE FILTERING ENABLED:")
    print(f"  Target file: {target_file if target_file else 'all'}")
    print(f"  Target event: {target_event if target_event is not None else 'all'}")
    if target_file_range is not None:
        print(f"  Target file range: file{target_file_range[0]}..file{target_file_range[1]} (inclusive)")

# ========================================================================
# SELECTION PARAMETERS
# ========================================================================
# Matching/efficiency/purity radii and the geometry-based cuts (fiducial YZ box,
# dead-area) are reused at the SAME values as the existing pipeline -- detector
# geometry hasn't changed and these are unit-independent of the new q/energy field.
radius_efficiency         = 2
radius_purity_xz          = 2
radius_purity_yz          = 5
radius_purity_xy          = 5
min_recopoints_threshold  = 5

# min_true_points_cutoff / min_reco_points_cutoff are DISABLED for now: this
# format's point clouds are much sparser than the old imaging-based
# reconstruction -- real neutrino clusters have been seen with as few as 13
# points -- so the old threshold (200) would delete real signal clusters
# outright. Revisit once correct values are known for this format's point
# density.
#
# min_cluster_energy IS applied (Apply_energy_cutoff = True): sed-sce's
# per-point 'e' field (MeV) is a genuine energy deposit -- same physical
# quantity/units as the old (non charge-light) pipeline's energy column --
# so the old threshold (100 MeV) carries over directly. See
# build_true_points_charge_light's energy= parameter (falls back to 'q'
# for older-format files that lack 'e').
min_cluster_energy        = 100     # APPLIED (Apply_energy_cutoff = True below)
min_true_points_cutoff    = 200     # NOT APPLIED (Apply_min_true_points_cutoff = False below)
min_reco_points_cutoff    = 200     # NOT APPLIED (Apply_min_reco_points_cutoff = False below)

# Apply selections
Apply_energy_cutoff                         = True    # sed-sce's 'e' field is genuine MeV -- see note above
Apply_min_true_points_cutoff                = False   # disabled -- see note above
Apply_min_reco_points_cutoff                = False   # disabled -- see note above
Apply_wire_readout_sensitive_xz_plane_cut   = True
Apply_time_window_cut                       = False   # must stay disabled -- no per-point true time in this format
Apply_deadarea_cut                          = True

# YZ Sensitivity cut parameters (detector geometry, unit: cm)
x_min = -250.0
x_max = 250.0
y_min = -200.0
y_max = 200.0
z_min = 0.15
z_max = 500.85

# Metadata label only (add_metadata_true_clusters/add_metadata_true_reco_pair_cluster
# store 'view' as a plain string field, not used for any logic) -- there's no
# 2-view/3-view distinction in the charge-light format, so this is just a constant.
view = "combined"

marker_size = 1

print("\nCuts applied:")
if Apply_energy_cutoff:
    print(f"- Energy cutoff applied (threshold {min_cluster_energy} MeV, using sed-sce's per-point 'e' field)")
if Apply_wire_readout_sensitive_xz_plane_cut:
    print(f"- Wire readout sensitive xz plane cut applied")
if Apply_deadarea_cut:
    print(f"- Dead area cut applied (split by X sign into APA0/APA1, see apply_deadarea_cut_true_charge_light)")
print("\nCuts not applied (see note above)")
if not Apply_min_true_points_cutoff:
    print(f"- Minimum true points cutoff not applied (threshold {min_true_points_cutoff} too aggressive for this format's point density)")
if not Apply_min_reco_points_cutoff:
    print(f"- Minimum reco points cutoff not applied (threshold {min_reco_points_cutoff} too aggressive for this format's point density)")

# ========================================================================
# ONE-TIME EXTRACTION
# ========================================================================
# Each file subdirectory ships as a single zip file. ensure_data_extracted()
# only unzips if that file's data/ folder doesn't already exist yet, so
# re-running this notebook never re-extracts.
if PARENT_DIR.exists():
    for subdir in sorted(PARENT_DIR.iterdir()):
        if subdir.is_dir():
            ensure_data_extracted(subdir)
else:
    print(f"Error: Parent directory {PARENT_DIR} does not exist")


In [ ]:
def find_all_input_directories(parent_dir):
    """
    Scan parent directory for all subdirectories containing 'data' folder.
    Returns a list of file directories (file0/, file1/, etc.).
    """
    parent_dir = Path(parent_dir)
    if not parent_dir.exists():
        print(f"Error: Parent directory {parent_dir} does not exist")
        return []

    data_dirs = []
    for subdir in sorted(parent_dir.iterdir()):
        if subdir.is_dir():
            data_path = subdir / "data"
            if data_path.exists() and data_path.is_dir():
                data_dirs.append(subdir)
                print(f"Found: {subdir}")

    return data_dirs

def file_index_from_name(name):
    """
    Trailing integer of a file directory name ("file10" -> 10), or None if it
    has no trailing digits. Used by target_file_range so files are selected by
    their real index rather than by position in the lexicographically sorted
    list (file0, file1, file10, file11, file2, ...).
    """
    digits = ""
    for ch in reversed(name):
        if not ch.isdigit():
            break
        digits = ch + digits
    return int(digits) if digits else None


def detect_events_in_directory(input_dir):
    """
    Auto-detect the number of events in a directory.
    Events are identified as numeric subdirectories in data/.
    Returns a sorted list of event numbers.
    """
    input_dir = Path(input_dir)
    data_dir = input_dir / "data"

    if not data_dir.exists():
        print(f"Warning: Data directory {data_dir} does not exist")
        return []

    events = []
    for item in data_dir.iterdir():
        if item.is_dir():
            try:
                event_num = int(item.name)
                events.append(event_num)
            except ValueError:
                pass

    return sorted(events)

# Auto-detect all input directories from parent directory
print(f"Scanning parent directory: {PARENT_DIR}")
print("-" * 60)
input_directories = find_all_input_directories(PARENT_DIR)
# Limit to num_files_to_process files for testing
if num_files_to_process is not None:
    input_directories = input_directories[:num_files_to_process]
else:
    num_files_to_process = len(input_directories)
print("-" * 60)

print(f"\nFound {len(input_directories)} input directories with data/\n")
if input_directories:
    for input_dir in input_directories:
        detected_events = detect_events_in_directory(input_dir)
        if detected_events:
            print(f"  {input_dir.name}/data/: {len(detected_events)} events ({min(detected_events)}-{max(detected_events)})")
        else:
            print(f"  {input_dir.name}/data/: No events found")
else:
    print(f"Error: No subdirectories with 'data/' found in {PARENT_DIR}")


In [ ]:
# ============================================================================
# MAIN LOOP -- count clusters surviving each successive cut
# ============================================================================
# Cuts are CUMULATIVE and applied in this order, mirroring the main pipeline's
# implementations exactly (same functions, same parameters from the config cell
# above -- nothing is reimplemented here):
#
#   true:  energy (min_cluster_energy)
#   reco:  beam window
#
# The true flow STARTS from the energy cut rather than from the raw uncut
# clusters: uncut, an event carries ~175 true clusters, almost all of them tiny
# sub-threshold depositions, which swamps every later stage (20356 clusters
# before the cut vs 76 after the beam window). Starting at the energy cut makes
# the first block the same population near_miss_investigation.py reports, so the
# two analyses' baselines agree. There is consequently no "no cuts" block, and
# the survival percentages are relative to the post-energy-cut counts.
#
# "Beam window" means the same thing it means everywhere else in this project:
# a true cluster is in the beam window if EvaluateEfficiency matches it (same
# radius_efficiency / min_recopoints_threshold) to one of the clustering-global
# clusters whose associated flash time falls inside
# [BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US]; a reco cluster is in the beam window
# if it IS one of those clusters. Reco clusters are grouped by real_cluster_id
# (not cluster_id, which can merge physically distinct tracks -- see the main
# notebook's header note).
#
# NOTE ON THE DEAD-AREA CUT FOR RECO: apply_deadarea_cut_true_charge_light is
# truth-named but is a purely GEOMETRIC row mask -- it drops points whose (y,z)
# falls inside a dead-channel polygon and touches no truth-only column -- so it
# applies unchanged to 5-column reco points. The main pipeline only ever runs it
# on the true side; applying it to reco here is deliberate, so that the reco flow
# has the same final geometry stage as the true flow and the two are comparable.
#
# The cut functions print per-cluster detail (hundreds of lines per event over a
# full job), so their stdout is suppressed -- the counts are what matter here.

timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = PLOTBASEDIR / f"selection_analysis_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"\n{'='*70}")
print(f"Output directory: {output_dir}")
print(f"{'='*70}\n")

# Stage definitions: (key, label, is_geometry_stage). is_geometry_stage marks the
# stages dropped from the "without geometry cuts" version of each plot.
# The dead-area / wire-readout stage is not tracked at all: measured over the full
# dataset it removed nothing from either side, so it only added a redundant row and
# a redundant block. The geometry cuts themselves are still APPLIED to the reco
# beam-window stage below, they are just not reported as a stage of their own.
TRUE_STAGES = [
    ('nocut',    'No cuts',                                   False),
    ('energy',   f'True energy >= {min_cluster_energy} MeV',   False),
]
RECO_STAGES = [
    ('nocut',    'No cuts',                                   False),
    ('beam',     '+ Beam window',                             False),
]

# Neutrinos are counted as mc.json INTERACTIONS split by vertex volume, not as
# clusters: at the no-cut stage an interaction that deposited nothing has no
# cluster to count, so a cluster-based tally would silently omit it (94 of 223
# over the full dataset). Cosmics stay cluster counts -- they have no mc.json
# vertex and no interaction concept.
job_true_counts = {key: {'cosmic': 0, 'neutrino_in': 0, 'neutrino_out': 0} for key, _, _ in TRUE_STAGES}
job_reco_counts = {key: {'total': 0} for key, _, _ in RECO_STAGES}
total_events_processed = 0
total_files_processed  = 0


def surviving_ids(points):
    """Distinct cluster_ids (column 3) left in a point array; empty set if none."""
    if points is None or len(points) == 0:
        return set()
    return set(np.unique(np.asarray(points)[:, 3]))


def points_of(clusters, keep_ids):
    """Stack the points of the clusters whose id is in keep_ids (empty -> None)."""
    kept = [np.asarray(pts) for cid, pts in clusters.items() if cid in keep_ids]
    return np.vstack(kept) if kept else None


for file_idx, input_dir in enumerate(input_directories):
    input_file_name = input_dir.name

    # SELECTIVE FILTERING: same selectors as the main notebook
    if target_file is not None and input_file_name != target_file:
        print(f"Skipping {input_file_name} (target: {target_file})")
        continue
    if target_file_range is not None:
        file_idx_parsed = file_index_from_name(input_file_name)
        range_low, range_high = target_file_range
        if file_idx_parsed is None or not (range_low <= file_idx_parsed <= range_high):
            print(f"Skipping {input_file_name} (target range: file{range_low}..file{range_high})")
            continue

    print(f"\n{'='*70}")
    print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir}")
    print(f"{'='*70}")

    events_list = detect_events_in_directory(input_dir)
    if not events_list:
        print(f"No events found in {input_dir}, skipping...")
        continue

    event_low = min(events_list)
    event_high = max(events_list) + 1 if num_events_to_process is None else event_low + num_events_to_process
    total_files_processed += 1

    for evt in range(event_low, event_high):
        if target_event is not None and evt != target_event:
            continue

        result = read_charge_light_files_for_event(input_dir, evt)
        if result is None:
            print(f"  Event {evt}: could not read data, skipping")
            continue
        event_key = f"{input_file_name}_{evt}"

        # ------------------------------------------------------------------
        # BEAM-WINDOW CLUSTERS (reco side), via the flash bridge
        # ------------------------------------------------------------------
        event_flash_metadata_list = build_cluster_flash_metadata(
            result['op'], input_file_name, evt, "Combined", event_key)
        event_img_cluster_flash_records = build_img_cluster_flash_metadata(
            result['reco'], result['clustering'], event_flash_metadata_list,
            input_file_name, evt, "Combined", event_key)

        x_clu, y_clu, z_clu, id_clu, q_clu, real_id_clu = result['clustering']
        reco_points_all  = np.column_stack((x_clu, y_clu, z_clu, real_id_clu, q_clu))
        clusters_all_clu = GroupClustersByID(reco_points_all)

        clu_beam_window_ids = {float(r['clustering_cluster_id']) for r in event_img_cluster_flash_records
                               if BEAM_WINDOW_MIN_US <= r['flash_time'] <= BEAM_WINDOW_MAX_US}
        clusters_clu_in_beam_window = {cid: pts for cid, pts in clusters_all_clu.items()
                                       if cid in clu_beam_window_ids}

        # ------------------------------------------------------------------
        # TRUE SIDE: build + reassign IDs (no cuts yet), then cut cumulatively
        # ------------------------------------------------------------------
        x_true, y_true, z_true, id_true, q_true, real_id_true, e_true, nu_idx_true = result['true_clustering']
        true_points_all = build_true_points_charge_light(
            x_true, y_true, z_true, id_true, q_true, energy=e_true, nu_idx=nu_idx_true)
        true_points_all = reassign_cluster_ID_true_charge_light(true_points_all)
        clusters_true_all = GroupClustersByID(true_points_all)

        # cluster_id -> is_neutrino, fixed once before any cut: IDs are assigned
        # by reassign_cluster_ID_true_charge_light BEFORE the cuts and the cuts
        # only ever drop points/clusters, never renumber, so this stays valid at
        # every stage below.
        is_neutrino_by_cid = {cid: bool(np.asarray(pts)[0, 4] > 0)
                              for cid, pts in clusters_true_all.items()}

        true_stage_ids = {'nocut': set(clusters_true_all.keys())}

        # energy cutoff (see header note: the population everything below is
        # compared against in practice, even though the raw uncut block is shown
        # above it for reference)
        true_points_stage = true_points_all
        if Apply_energy_cutoff:
            true_points_stage = apply_energy_cutoff(true_points_stage, min_cluster_energy)
        true_stage_ids['energy'] = surviving_ids(true_points_stage)

        # NO beam-window stage on the true side: a true cluster carries no flash
        # and no time, so "in beam window" could only be inferred by matching it
        # to a beam-window-flashed reco cluster -- mixing beam timing with
        # reconstruction efficiency. Beam window stays reco-side only (below).

        # ------------------------------------------------------------------
        # NEUTRINO INTERACTIONS (mc.json) -- the neutrino side of the flow.
        # Joined to true clusters by nu_idx (cluster_id = 99990+nu_idx), so a
        # stage's neutrino count is "interactions whose cluster survived that
        # stage", and the no-cut stage is simply every interaction in mc.json.
        # ------------------------------------------------------------------
        event_vertex_records = build_neutrino_vertex_records(
            flatten_mc_tree(result['mc']), clusters_true_all, input_file_name, evt, event_key,
            x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max, z_min=z_min, z_max=z_max)

        for key, _, _ in TRUE_STAGES:
            ids = true_stage_ids[key]
            job_true_counts[key]['cosmic'] += sum(
                1 for cid in ids if not is_neutrino_by_cid.get(cid, False))
            for record in event_vertex_records:
                # no-cut stage counts every interaction; later stages only those
                # whose cluster is still present
                if key != 'nocut' and record['cluster_id'] not in ids:
                    continue
                if record['vertex_in_volume'] is True:
                    job_true_counts[key]['neutrino_in'] += 1
                else:
                    job_true_counts[key]['neutrino_out'] += 1

        # ------------------------------------------------------------------
        # RECO SIDE: no truth label, total only
        # ------------------------------------------------------------------
        reco_stage_ids = {
            'nocut': set(clusters_all_clu.keys()),
            'beam':  set(clusters_clu_in_beam_window.keys()),
        }
        for key, _, _ in RECO_STAGES:
            job_reco_counts[key]['total'] += len(reco_stage_ids[key])

        total_events_processed += 1
        n_nu_energy = sum(1 for r in event_vertex_records if r['cluster_id'] in true_stage_ids['energy'])
        print(f"  {event_key}: true clusters {len(true_stage_ids['nocut'])} -> energy {len(true_stage_ids['energy'])}"
              f"   |   mc neutrinos {len(event_vertex_records)} -> energy {n_nu_energy}"
              f"   |   reco {len(reco_stage_ids['nocut'])} -> beam {len(reco_stage_ids['beam'])}", flush=True)

print(f"\n{'='*70}")
print(f"JOB SUMMARY: {total_files_processed} file(s), {total_events_processed} event(s) processed")
print(f"{'='*70}")

In [ ]:
# ============================================================================
# DRAW
# ============================================================================
# Every stage is still COUNTED and still written to selection_flow_counts.txt --
# only the drawn bars are reduced:
#   true plot: No cuts + energy cut. There is no true-side beam-window stage at
#     all any more -- "in beam window" is not a truth quantity (true clusters
#     carry no flash and no time), so it is neither counted nor drawn here. The
#     reco plot keeps its beam-window block, where the flash time is measured.
#   both plots: the dead-area / wire-readout block is not drawn (it removed
#     nothing at either level -- the counts are in the table if needed).
true_stage_records = [
    {'key': key, 'stage': label, 'geometry': is_geometry, **job_true_counts[key]}
    for key, label, is_geometry in TRUE_STAGES
]
reco_stage_records = [
    {'key': key, 'stage': label, 'geometry': is_geometry, **job_reco_counts[key]}
    for key, label, is_geometry in RECO_STAGES
]

true_plot_records = list(true_stage_records)
reco_plot_records = list(reco_stage_records)

# include_geometry_cuts=False so the filename/title say so honestly -- the records
# passed in already carry no geometry stage, making the drawer's own filter a no-op.
DrawTrueClusterSelectionFlow(true_plot_records, output_dir, "Job Level", "job", "Combined",
                             include_geometry_cuts=False)
DrawRecoClusterSelectionFlow(reco_plot_records, output_dir, "Job Level", "job", "Combined",
                             include_geometry_cuts=False)

table_path = write_selection_flow_table(true_stage_records, reco_stage_records, output_dir,
                                        level_name="Job Level")

print("TRUE selection flow (cosmic clusters | neutrino interactions by vertex volume):")
for r in true_stage_records:
    n_nu = r['neutrino_in'] + r['neutrino_out']
    print(f"  {r['stage']:<32} cosmic={r['cosmic']:>6}  neutrino={n_nu:>5} "
          f"(in-volume {r['neutrino_in']}, out-volume {r['neutrino_out']})")
reco_drawn = ", ".join(r['stage'] for r in reco_plot_records)
print(f"\nRECO cluster selection flow (drawn: {reco_drawn}):")
for r in reco_stage_records:
    print(f"  {r['stage']:<32} total={r['total']:>6}")

print(f"\nPlots written to: {output_dir}")
print(f"Counts table written to: {table_path}")

# ============================================================================
# SUMMARY (job level): configuration, what was processed, the flow counts, and
# the job's start/finish/runtime -- the same JOB RUNTIME section the main
# notebook's job_summary/summary.txt carries, so the two are comparable.
# ============================================================================
job_finish_time = time.time()
job_finish_dt   = datetime.now()
job_runtime     = job_finish_time - job_start_time

summary_lines = []
summary_lines.append("=" * 80)
summary_lines.append("SELECTION ANALYSIS SUMMARY")
summary_lines.append("=" * 80)
summary_lines.append(f"Generated: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append("")
summary_lines.append("Configuration:")
summary_lines.append(f"  Parent directory: {PARENT_DIR}")
summary_lines.append(f"  Files to process:  {files}")
summary_lines.append(f"  Events to process: {events}")
summary_lines.append(f"  target_file: {target_file}, target_event: {target_event}, "
                     f"target_file_range: {target_file_range}")
summary_lines.append(f"  Energy cut: {min_cluster_energy} MeV (applied: {Apply_energy_cutoff})")
summary_lines.append(f"  Volume (wire-readout sensitive box): "
                     f"x [{x_min:g}, {x_max:g}], y [{y_min:g}, {y_max:g}], z [{z_min:g}, {z_max:g}] cm")
summary_lines.append("")
summary_lines.append(f"Files processed:  {total_files_processed}")
summary_lines.append(f"Events processed: {total_events_processed}")
summary_lines.append("")

summary_lines.append("=" * 80)
summary_lines.append("TRUE SELECTION FLOW")
summary_lines.append("=" * 80)
summary_lines.append("cosmic   = cosmic CLUSTERS surviving the stage")
summary_lines.append("neutrino = true neutrino INTERACTIONS from mc.json, split by vertex volume")
for r in true_stage_records:
    n_nu = r['neutrino_in'] + r['neutrino_out']
    summary_lines.append(f"  {r['stage']:<34} cosmic={r['cosmic']:>7}  neutrino={n_nu:>6} "
                         f"(in-volume {r['neutrino_in']}, out-volume {r['neutrino_out']})")
summary_lines.append("")
summary_lines.append("=" * 80)
summary_lines.append("RECO SELECTION FLOW")
summary_lines.append("=" * 80)
for r in reco_stage_records:
    summary_lines.append(f"  {r['stage']:<34} total={r['total']:>7}")
summary_lines.append("")

summary_lines.append("=" * 80)
summary_lines.append("JOB RUNTIME")
summary_lines.append("=" * 80)
summary_lines.append(f"Job started at:  {datetime.fromtimestamp(job_start_time).strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Job finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Total job runtime: {timedelta(seconds=int(job_runtime))} ({job_runtime:.1f} seconds)")
summary_lines.append("=" * 80)

with open(output_dir / "summary.txt", "w") as f:
    f.write("\n".join(summary_lines) + "\n")

print(f"\nJob finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')} (runtime: {job_runtime:.1f}s)")
print(f"Summary written to: {output_dir / 'summary.txt'}")